# Creating your own dataset

In this exercise, we'll use the GitHub issues associated with a popular open source project: 🤗 Datasets! Let's take a look at how to get the data and explore the information contained in these issues.

### Getting the data

You can find all the issues in 🤗 Datasets by navigating to the repository's [Issues tab](https://github.com/huggingface/datasets/issues).

<div class="flex justify-center">
<img src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter5/datasets-issues.png" alt="The GitHub issues associated with 🤗 Datasets." width="40%"/>
</div>

If you click on one of these issues you'll find it contains a title, a description, and a set of labels that characterize the issue. An example is shown in the screenshot below.

<div class="flex justify-center">
<img src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter5/datasets-issues-single.png" alt="A typical GitHub issue in the 🤗 Datasets repository." width="40%"/>
</div>

To download all the repository's issues, we'll use the [GitHub REST API](https://docs.github.com/en/rest) to poll the [`Issues` endpoint](https://docs.github.com/en/rest/reference/issues#list-repository-issues). This endpoint returns a list of JSON objects, with each object containing a large number of fields that include the title and description as well as metadata about the status of the issue and so on.

In [ ]:
# %pip install datasets evaluate transformers[sentencepiece]
# !apt install git-lfs

You will need to setup git, adapt your email and name in the following cell. <br>

In [ ]:
!git config --global user.email "you@example.com" # I entered my mail-id here
!git config --global user.name "Your Name" # I entered my name here

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.<br>

Here you need to enter your HuggingFace access token. If you do not have an access token already, you need to create it. I have described the steps in the [README](README.md#creating-a-huggingface-access-token).

In [2]:
from huggingface_hub import notebook_login

notebook_login()

A convenient way to download the issues is via the `requests` library, which is the standard way for making HTTP requests in Python. You can install the library by running:

In [ ]:
# !pip install requests

Once the library is installed, we can make GET requests to the `Issues` endpoint by invoking the `requests.get()` function. For example, we can run the following command to retrieve the first issue on the first page:

In [3]:
import requests

url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url)

The `response` object contains a lot of useful information about the request, including the HTTP status code:

In [4]:
response.status_code

200

where a `200` status means the request was successful (you can find a list of possible HTTP status codes [here](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes)). What we are really interested in, though, is the _payload_, which can be accessed in various formats like bytes, strings, or JSON. Since we know our issues are in JSON format, let's inspect the payload as follows:


In [5]:
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/8501',
  'repository_url': 'https://api.github.com/repos/huggingface/datasets',
  'labels_url': 'https://api.github.com/repos/huggingface/datasets/issues/8501/labels{/name}',
  'comments_url': 'https://api.github.com/repos/huggingface/datasets/issues/8501/comments',
  'events_url': 'https://api.github.com/repos/huggingface/datasets/issues/8501/events',
  'html_url': 'https://github.com/huggingface/datasets/pull/8501',
  'id': 5198172545,
  'node_id': 'PR_kwDODunzps8AAAABAWX4Qw',
  'number': 8501,
  'title': 'Format bool and temporal values as numpy scalars and keep the duration dtype in numpy format',
  'user': {'login': '2sumtech',
   'id': 206164974,
   'node_id': 'U_kgDODEnT7g',
   'avatar_url': 'https://avatars.githubusercontent.com/u/206164974?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/2sumtech',
   'html_url': 'https://github.com/2sumtech',
   'followers_url': 'https://api.github.com/users

That's a lot of information! We can see useful fields like `title`, `body`, and `number` that describe the issue, as well as information about the GitHub user who opened the issue.

As described in the GitHub [documentation](https://docs.github.com/en/rest/overview/resources-in-the-rest-api#rate-limiting), unauthenticated requests are limited to 60 requests per hour. Although we can increase the `per_page` query parameter to reduce the number of requests we make, we will still hit the rate limit on any repository that has more than a few thousand issues. So instead, we should follow GitHub's [instructions](https://docs.github.com/en/github/authenticating-to-github/creating-a-personal-access-token) on creating a _personal access token_ so that we can boost the rate limit to 5,000 requests per hour. Once we have our token, we can include it as part of the request header:

In [ ]:
# GITHUB_TOKEN = xxx  # Copy your GitHub token here
# headers = {"Authorization": f"token {GITHUB_TOKEN}"}

In [ ]:
# import time
# import math
# from pathlib import Path
# import pandas as pd
# from tqdm.notebook import tqdm


# def fetch_issues(
#     owner="huggingface",
#     repo="datasets",
#     num_issues=10_000,
#     rate_limit=5_000,
#     issues_path=Path("."),
# ):
#     if not issues_path.is_dir():
#         issues_path.mkdir(exist_ok=True)

#     batch = []
#     all_issues = []
#     per_page = 100  # Number of issues to return per page
#     num_pages = math.ceil(num_issues / per_page)
#     base_url = "https://api.github.com/repos"

#     for page in tqdm(range(num_pages)):
#         # Query with state=all to get both open and closed issues
#         query = f"issues?page={page}&per_page={per_page}&state=all"
#         issues = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)
#         batch.extend(issues.json())

#         if len(batch) > rate_limit and len(all_issues) < num_issues:
#             all_issues.extend(batch)
#             batch = []  # Flush batch for next time period
#             print(f"Reached GitHub rate limit. Sleeping for one hour ...")
#             time.sleep(60 * 60 + 1)

#     all_issues.extend(batch)
#     df = pd.DataFrame.from_records(all_issues)
#     df.to_json(f"{issues_path}/{repo}-issues.jsonl", orient="records", lines=True)
#     print(f"Downloaded all the issues for {repo}! Dataset stored at {issues_path}/{repo}-issues.jsonl")

The above two commented out cells contain the original course material's code. I have changed them a bit.

>### Changes from the original Hugging Face course code
>
>1. Authentication - Changed the GitHub header to use the current `Bearer` format.
>2. Error handling - Added an HTTP status check so GitHub API errors such as `HTTP 422` are clearly displayed.
>3. Rate-limit handling - Corrected the logic so that **API requests**, rather than the number of downloaded records, are considered when handling the rate limit.
>4. Data/API behavior - Kept the GitHub REST Issues API, `state="all"`, and `per_page=100` so the dataset remains consistent with the Hugging Face course.
>5. Output - Kept the original JSONL format: `datasets-issues.jsonl`
>
>These changes improve authentication, error handling, and rate-limit handling without changing the overall purpose of the course exercise.


Also, I had to **create a GitHub access token** for this part. The steps for doing this are mentioned in the [README](README.md).

In [ ]:
GITHUB_TOKEN = "-------------------------------------"  # Copy your GitHub token here
headers = {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github+json"}

Now that we have our access token, let's create a function that can download all the issues from a GitHub repository:

In [24]:
import time
import math
from pathlib import Path

import pandas as pd
import requests
from tqdm.notebook import tqdm


def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=10_000,
    rate_limit=5_000,
    issues_path=Path("."),
):
    """
    Download GitHub issues using the Hugging Face course approach.

    Modification from the original course:
    The original code compares the number of downloaded records
    with the API request limit. We instead estimate the number
    of requests made using the number of pages downloaded.

    This avoids unnecessarily sleeping after 5,000 records.
    """

    if not issues_path.is_dir():
        issues_path.mkdir(parents=True, exist_ok=True)

    batch = []
    all_issues = []

    per_page = 100
    num_pages = math.ceil(num_issues / per_page)

    base_url = "https://api.github.com/repos"

    for page in tqdm(range(num_pages)):

        query = f"issues?page={page + 1}" f"&per_page={per_page}" f"&state=all"

        response = requests.get(
            f"{base_url}/{owner}/{repo}/{query}",
            headers=headers,
        )

        # Check for API errors
        if response.status_code != 200:
            print(f"\nGitHub returned HTTP " f"{response.status_code}")
            print(response.text)
            break

        issues = response.json()

        batch.extend(issues)

        # ------------------------------------------------
        # IMPORTANT:
        #
        # We have made approximately one API request
        # per page, NOT one request per issue.
        # ------------------------------------------------

        requests_made = page + 1

        if requests_made >= rate_limit and len(all_issues) < num_issues:
            print("\nGitHub API rate limit reached. " "Sleeping for one hour...")

            time.sleep(60 * 60 + 1)

            all_issues.extend(batch)
            batch = []

        elif len(batch) >= per_page:
            all_issues.extend(batch)
            batch = []

        # Stop once we have enough
        if len(all_issues) >= num_issues:
            break

    # Add anything left in batch
    all_issues.extend(batch)

    # Don't exceed requested amount
    all_issues = all_issues[:num_issues]

    # Convert to DataFrame
    df = pd.DataFrame.from_records(all_issues)

    # Save JSONL
    output_file = issues_path / f"{repo}-issues.jsonl"

    df.to_json(
        output_file,
        orient="records",
        lines=True,
    )

    print("\n----------------------------------")
    print(f"Downloaded {len(df)} issues/PRs for {repo}!")
    print(f"Dataset stored at: " f"{output_file.resolve()}")
    print("----------------------------------")

Now when we call `fetch_issues()` it will download all the issues in batches to avoid exceeding GitHub's limit on the number of requests per hour; the result will be stored in a _repository_name-issues.jsonl_ file, where each line is a JSON object the represents an issue. Let's use this function to grab all the issues from 🤗 Datasets:

In [ ]:
fetch_issues()

```text
GitHub returned HTTP 422
{"message":"Pagination with the page parameter is not supported for large datasets, please use cursor based pagination (after/before)","documentation_url":"https://docs.github.com/rest/issues/issues#list-repository-issues","status":"422"}

----------------------------------
Downloaded 8281 issues/PRs for datasets!
```

> **Why the download stopped at 8,281 records?**

> The course requests up to 10,000 records using GitHub's REST API with page-based pagination.
> Our download successfully retrieved 8,281 records. When the code attempted to request another page, GitHub returned:
>
> _HTTP 422: "Pagination with the page parameter is not supported for large datasets, please use cursor based pagination (after/before)"_
>
> This is a current GitHub API pagination limitation, not a problem with our GitHub account, token, permissions, or rate limit.
>
> The 8,281 records were successfully downloaded and saved to: `datasets-issues.jsonl`
> We can continue using the 8,281 records.

Let's check how many requests we are left with:

In [27]:
test_response = requests.get("https://api.github.com/user", headers=headers)

print("Status:", test_response.status_code)
print("Authenticated as:", test_response.json().get("login"))
print("Rate limit:", test_response.headers.get("X-RateLimit-Limit"))
print("Remaining:", test_response.headers.get("X-RateLimit-Remaining"))

Status: 200
Authenticated as: avirupc
Rate limit: 5000
Remaining: 4898


Once the issues are downloaded we can load them locally:

In [3]:
from datasets import load_dataset

In [37]:
# issues_dataset = load_dataset("json", data_files="datasets-issues.jsonl", split="train")
# issues_dataset

The above code was provided in the course notebook, but it threw an error
because the current GitHub dataset contains inconsistent data types in
some fields. Therefore, I used the workaround below, which basically
converts the JSONL file into a regular JSON file first and then loads it
using the Hugging Face `load_dataset()` function. This avoids the schema
inference issue encountered when loading the JSONL file directly.

In [4]:
import json

# Read the existing JSONL file
with open("datasets-issues.jsonl", "r", encoding="utf-8") as f:
    lines = [json.loads(line) for line in f]

# Save as a regular JSON array
with open("datasets-issues.json", "w", encoding="utf-8") as f:
    json.dump(lines, f, indent=2)

print(f"Converted {len(lines)} records to datasets-issues.json")

Converted 8281 records to datasets-issues.json


In [5]:
from datasets import load_dataset

issues_dataset = load_dataset("json", data_files="datasets-issues.json", split="train")

issues_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment'],
    num_rows: 8281
})

##  Cleaning up the data

One thing to note here:

> GitHub's REST API v3 considers every pull request an issue, but not every issue is a pull request. For this reason, "Issues" endpoints may return both issues and pull requests in the response. You can identify pull requests by the `pull_request` key. Be aware that the `id` of a pull request returned from "Issues" endpoints will be an issue id.

Since the contents of issues and pull requests are quite different, let's do some minor preprocessing to enable us to distinguish between them.

The above snippet from GitHub's documentation tells us that the `pull_request` column can be used to differentiate between issues and pull requests. Let's look at a random sample to see what the difference is. As we did in section 3, we'll chain `Dataset.shuffle()` and `Dataset.select()` to create a random sample and then zip the `html_url` and `pull_request` columns so we can compare the various URLs:

In [6]:
sample = issues_dataset.shuffle(seed=666).select(range(3))

# Print out the URL and pull request entries
for url, pr in zip(sample["html_url"], sample["pull_request"]):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}\n")

>> URL: https://github.com/huggingface/datasets/issues/206
>> Pull request: None

>> URL: https://github.com/huggingface/datasets/pull/3278
>> Pull request: {'diff_url': 'https://github.com/huggingface/datasets/pull/3278.diff', 'html_url': 'https://github.com/huggingface/datasets/pull/3278', 'merged_at': '2021-11-16T11:19:37Z', 'patch_url': 'https://github.com/huggingface/datasets/pull/3278.patch', 'url': 'https://api.github.com/repos/huggingface/datasets/pulls/3278'}

>> URL: https://github.com/huggingface/datasets/pull/1874
>> Pull request: {'diff_url': 'https://github.com/huggingface/datasets/pull/1874.diff', 'html_url': 'https://github.com/huggingface/datasets/pull/1874', 'merged_at': '2021-03-04T10:38:22Z', 'patch_url': 'https://github.com/huggingface/datasets/pull/1874.patch', 'url': 'https://api.github.com/repos/huggingface/datasets/pulls/1874'}



Here we can see that each pull request is associated with various URLs, while ordinary issues have a `None` entry. We can use this distinction to create a new `is_pull_request` column that checks whether the `pull_request` field is `None` or not:

In [38]:
issues_dataset = issues_dataset.map(lambda x: {"is_pull_request": False if x["pull_request"] is None else True})

Map:   0%|          | 0/8281 [00:00<?, ? examples/s]

Although we could proceed to further clean up the dataset by dropping or renaming some columns, it is generally a good practice to keep the dataset as "raw" as possible at this stage so that it can be easily used in multiple applications.

Before we push our dataset to the Hugging Face Hub, let's deal with one thing that's missing from it: the comments associated with each issue and pull request. We'll add them next with, again, the GitHub REST API!


## Augmenting the dataset

As shown in the following screenshot, the comments associated with an issue or pull request provide a rich source of information, especially if we're interested in building a search engine to answer user queries about the library.

<div class="flex justify-center">
<img src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter5/datasets-issues-comment.png" alt="Comments associated with an issue about 🤗 Datasets." width="40%"/>
</div>

The GitHub REST API provides a [`Comments` endpoint](https://docs.github.com/en/rest/reference/issues#list-issue-comments) that returns all the comments associated with an issue number. Let's test the endpoint to see what it returns:

In [ ]:
issue_number = 2792
url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
response = requests.get(url, headers=headers)
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/897594128',
  'html_url': 'https://github.com/huggingface/datasets/pull/2792#issuecomment-897594128',
  'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/2792',
  'id': 897594128,
  'node_id': 'IC_kwDODunzps41gDMQ',
  'user': {'login': 'bhavitvyamalik',
   'id': 19718818,
   'node_id': 'MDQ6VXNlcjE5NzE4ODE4',
   'avatar_url': 'https://avatars.githubusercontent.com/u/19718818?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/bhavitvyamalik',
   'html_url': 'https://github.com/bhavitvyamalik',
   'followers_url': 'https://api.github.com/users/bhavitvyamalik/followers',
   'following_url': 'https://api.github.com/users/bhavitvyamalik/following{/other_user}',
   'gists_url': 'https://api.github.com/users/bhavitvyamalik/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/bhavitvyamalik/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/

We can see that the comment is stored in the `body` field, so let's write a simple function that returns all the comments associated with an issue by picking out the `body` contents for each element in `response.json()`:

In [11]:
def get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    response = requests.get(url, headers=headers)
    return [r["body"] for r in response.json()]


# Test our function works as expected
get_comments(2792)

["@albertvillanova my tests are failing here:\r\n```\r\ndataset_name = 'gooaq'\r\n\r\n    def test_load_dataset(self, dataset_name):\r\n        configs = self.dataset_tester.load_all_configs(dataset_name, is_local=True)[:1]\r\n>       self.dataset_tester.check_load_dataset(dataset_name, configs, is_local=True, use_local_dummy_data=True)\r\n\r\ntests/test_dataset_common.py:234: \r\n_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ \r\ntests/test_dataset_common.py:187: in check_load_dataset\r\n    self.parent.assertTrue(len(dataset[split]) > 0)\r\nE   AssertionError: False is not true\r\n```\r\nWhen I try loading dataset on local machine it works fine. Any suggestions on how can I avoid this error?",
 'Thanks for the help, @albertvillanova! All tests are passing now.']

This looks good, so let's use `Dataset.map()` to add a new `comments` column to each issue in our dataset:

In [ ]:
# Depending on your internet connection, this can take a few minutes...
issues_with_comments_dataset = issues_dataset.map(lambda x: {"comments": get_comments(x["number"])})


# This cell took a lot of time to finish running, almost 2 hours!

Map:   0%|          | 0/8281 [00:00<?, ? examples/s]

In [ ]:
# Sabed this to a new json
issues_with_comments_dataset.to_json("datasets-issues_with_comments.json")

Creating json from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

60916926

I had to clean the newly created json file once again as there were some empty fields, and without doing this I was getting an error in the next chapter's [notebook](../5.6%20Semantic%20Search%20with%20FAISS/Semantic_search_with_FAISS_(PyTorch).ipynb).

In [27]:
import json

# Read the existing JSONL file
with open("datasets-issues_with_comments.json", "r", encoding="utf-8") as f:
    lines = [json.loads(line) for line in f]

# Save as a regular JSON array
with open("datasets-issues_with_comments_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(lines, f, indent=2)

print(f"Converted {len(lines)} records to datasets-issues_with_comments_cleaned.json")

Converted 8281 records to datasets-issues_with_comments_cleaned.json


In [28]:
test_response = requests.get("https://api.github.com/user", headers=headers)

print("Status:", test_response.status_code)
print("Authenticated as:", test_response.json().get("login"))
print("Rate limit:", test_response.headers.get("X-RateLimit-Limit"))
print("Remaining:", test_response.headers.get("X-RateLimit-Remaining"))

Status: 200
Authenticated as: avirupc
Rate limit: 5000
Remaining: 4999


The final step is to push our dataset to the Hub. Let’s take a look at how we can do that.

_I did not try the cells onward. If you want more detailed documentation of these steps, check the descriptions along with the code cells from the [chapter](https://huggingface.co/learn/llm-course/chapter5/5#uploading-the-dataset-to-the-hugging-face-hub)_

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
issues_with_comments_dataset.push_to_hub("github-issues")

In [ ]:
remote_dataset = load_dataset("lewtun/github-issues", split="train")
remote_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 2855
})